In [70]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from helpers import missing_summary

cwd = Path.cwd()
DATA_DIR = Path(cwd).parent / "data"

# Import data

In [55]:
training_data = pd.read_csv(DATA_DIR / "train.csv")

x = training_data.drop(columns="Survived")

y = training_data["Survived"]

In [56]:
x.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [57]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Name         891 non-null    str    
 3   Sex          891 non-null    str    
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    str    
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    str    
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 76.7 KB


# Data cleaning

In [58]:
# First, get an overview of missing variables

missing_summary(x)

,dtype,missing_count,missing_pct,unique_values
Cabin,str,687,77.104377,147
Age,float64,177,19.865320,88
Embarked,str,2,0.224467,3
Name,str,0,0.000000,891
Pclass,int64,0,0.000000,3
PassengerId,int64,0,0.000000,891
Sex,str,0,0.000000,2
Parch,int64,0,0.000000,7
SibSp,int64,0,0.000000,7
Fare,float64,0,0.000000,248


In [59]:
# Check if there are some pattern to missing variables in age. Maybe something can be inferred from another variable. Age could be inferred from title of name, ticket class, cabin, parents or ticket
x[x['Age'].isna()]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
5,6,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
17,18,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S
19,20,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
26,27,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
28,29,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
...,...,...,...,...,...,...,...,...,...,...,...
859,860,3,"Razi, Mr. Raihed",male,NaN,0,0,2629,7.2292,NaN,C
863,864,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S
868,869,3,"van Melkebeke, Mr. Philemon",male,NaN,0,0,345777,9.5000,NaN,S
878,879,3,"Laleff, Mr. Kristo",male,NaN,0,0,349217,7.8958,NaN,S


In [60]:
# Check wether survivors are more likely to have missing age or cabin variables

training_data.assign(
    age_missing=training_data["Age"].isna()
).groupby("age_missing")["Survived"].agg(
    ["count", "mean"]
)

,count,mean
age_missing,,
False,714,0.406162
True,177,0.293785


In [61]:
training_data.assign(
    cabin_missing=training_data["Cabin"].isna()
).groupby("cabin_missing")["Survived"].agg(
    ["count", "mean"]
)

,count,mean
cabin_missing,,
False,204,0.666667
True,687,0.299854


In [62]:
# Pclass represent ticket class for passengers. Turn into categorical variable

bins =  [0,1,2,3]

labels = ["1st", "2nd", "3nd"]

x["ticket_class"] = pd.cut(x["Pclass"], bins=bins, labels=labels) # Unecessary complicated

# Alternative

x['ticket_class'] = x["Pclass"].map({
    1 : "1st",
    2 : "2nd",
    3 : "3nd"
})


In [63]:
x['has_cabin'] = x['Cabin'].notna()

In [79]:
# However. Scikit-learn provides a Pipeline object that can manage preprocessing of data - like categorising the data

categorical_features = ["Sex", "Pclass", "Embarked", "ticket_class", "has_cabin"]
numerical_features = ['Age', 'SibSp', 'Parch'] # Age has missing values. To start with, it will be imputed with median value. Infer age later

categorical_pipeline = Pipeline([
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    ),
    (
        "numerical",
        numerical_pipeline,
        numerical_features,
    ),
    
])

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(max_iter=1000)
    )
])

In [80]:
model.fit(x, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['PassengerId','Pclass','Name',...,'Embarked','ticket_class','has_cabin']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder=

In [81]:
log_reg = model.named_steps["classifier"]

log_reg.coef_

array([[ 1.3280651 , -1.33054902,  0.30199818,  0.14576517, -0.45024727,
         0.14732731,  0.04490966, -0.35555321,  0.16083233,  0.30199818,
         0.14576517, -0.45024727, -0.4851378 ,  0.48265388, -0.03848522,
        -0.29762817, -0.08775207]])

In [82]:
preprocessor = model.named_steps["preprocessor"]
log_reg = model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg.coef_[0],
})

coefficients.sort_values("coefficient", ascending=False)

,feature,coefficient
0,categorical__Sex_female,1.328065
13,categorical__has_cabin_True,0.482654
2,categorical__Pclass_1,0.301998
9,categorical__ticket_class_1st,0.301998
8,categorical__Embarked_nan,0.160832
5,categorical__Embarked_C,0.147327
10,categorical__ticket_class_2nd,0.145765
3,categorical__Pclass_2,0.145765
6,categorical__Embarked_Q,0.044910
14,numerical__Age,-0.038485


In [83]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scores = cross_val_score(
    model,
    x,
    y,
    cv=cv,
    scoring="accuracy",
)

print("Accuracy per fold:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

Accuracy per fold: [0.81005587 0.81460674 0.76966292 0.79213483 0.84269663]
Mean accuracy: 0.8058313979034587
Std: 0.024288485727769264
